# Language Models in DemoGPT

DemoGPT provides convenient wrappers around popular Large Language Models (LLMs) through the `demogpt_agenthub.llms` module. These wrappers offer a unified interface for interacting with different LLM providers, making it easy to swap models, configure parameters, and integrate them into agents.

In this notebook, we will explore:
- How to use **OpenAIChatModel** for chat-based completions
- How to configure model parameters like temperature and max tokens
- How to use **OpenAIModel** for text completions
- How to point models at custom API endpoints
- How to pass LLMs to DemoGPT agents

## Setting Up Your API Key

Create a `.env` file in your project root with your OpenAI API key:

```
OPENAI_API_KEY=sk-your-key-here
```

Then load it with `python-dotenv` (already included with DemoGPT):

In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")  # Loads OPENAI_API_KEY from .env file

import os
print("API key loaded:", "OPENAI_API_KEY" in os.environ)

## 2. OpenAIChatModel (Chat Completion)

`OpenAIChatModel` wraps the OpenAI chat completion API (used by models like GPT-4o and GPT-4.1). It extends LangChain's `ChatOpenAI` and DemoGPT's `BaseLLM`, providing a simple `.run()` method that sends a prompt and returns the model's text response.

This is the recommended model class for most use cases.

In [ ]:
from demogpt_agenthub.llms import OpenAIChatModel

# Create an instance with gpt-4o-mini
llm = OpenAIChatModel(model_name="gpt-4o-mini")

# Run a simple prompt
response = llm.run("What is Python in one sentence?")
print(response)

## 3. Model Parameters

You can configure common model parameters when creating an `OpenAIChatModel` instance:

- **temperature**: Controls randomness. Lower values (e.g., 0.0) produce more deterministic output; higher values (e.g., 1.0) produce more creative output.
- **max_tokens**: Limits the maximum number of tokens in the response.
- **api_key**: Pass your API key directly instead of using an environment variable.

In [ ]:
# Deterministic output (temperature=0)
llm_deterministic = OpenAIChatModel(
    model_name="gpt-4o-mini",
    temperature=0.0,
    max_tokens=100,
)

response_deterministic = llm_deterministic.run("Write a one-line tagline for a coffee shop.")
print("Deterministic (temperature=0.0):")
print(response_deterministic)
print()

# Creative output (temperature=1.0)
llm_creative = OpenAIChatModel(
    model_name="gpt-4o-mini",
    temperature=1.0,
    max_tokens=100,
)

response_creative = llm_creative.run("Write a one-line tagline for a coffee shop.")
print("Creative (temperature=1.0):")
print(response_creative)

In [ ]:
# You can also pass the API key directly
llm_with_key = OpenAIChatModel(
    model_name="gpt-4o-mini",
    api_key="sk-your-api-key-here",  # pass key directly
    temperature=0.5,
    max_tokens=200,
)

# This is useful when managing multiple API keys or
# when you don't want to rely on environment variables.

## 4. OpenAIModel (Legacy Text Completion)

`OpenAIModel` wraps the OpenAI text completion API. This is intended for older completion-style models like `gpt-3.5-turbo-instruct` that take a text prompt and return a continuation, rather than operating in a chat format.

> **Warning:** OpenAI's text completion models are **legacy** and deprecated. The `gpt-3.5-turbo-instruct` model is the last remaining completion model and may be removed in the future. For all new projects, use `OpenAIChatModel` with a modern chat model like `gpt-4o-mini` or `gpt-4.1`.

In [ ]:
from demogpt_agenthub.llms import OpenAIModel

# Create a text completion model (legacy - prefer OpenAIChatModel for new projects)
text_llm = OpenAIModel(model_name="gpt-3.5-turbo-instruct")

# Run a text completion prompt
response = text_llm.run("The three primary colors are")
print(response)

## 5. Available Models

DemoGPT's OpenAI wrappers support any model available through the OpenAI API. Here are the current recommended models:

| Model | Type | Notes |
|-------|------|-------|
| `gpt-4.1` | Chat | Most capable, latest model |
| `gpt-4.1-mini` | Chat | Balanced performance and cost |
| `gpt-4.1-nano` | Chat | Fastest, lowest cost |
| `gpt-4o` | Chat | Strong multimodal model |
| `gpt-4o-mini` | Chat | Compact and cost-effective |
| `o3` | Reasoning | Best for complex reasoning tasks |
| `o4-mini` | Reasoning | Cost-effective reasoning model |

> **Note:** Older models like `gpt-3.5-turbo`, `gpt-3.5-turbo-16k`, `gpt-4`, and `gpt-4-turbo` are deprecated. Use `gpt-4o-mini` as a drop-in replacement for GPT-3.5 models, and `gpt-4o` as a replacement for GPT-4 models.

Use **`OpenAIChatModel`** for all chat and reasoning models. **`OpenAIModel`** is only needed for the legacy `gpt-3.5-turbo-instruct` completion model.

## 6. Custom API Base URL

If you are running a self-hosted model server (e.g., vLLM, Ollama with OpenAI-compatible API, or a proxy), you can point DemoGPT's LLM wrappers at a custom API base URL using the `base_url` parameter.

In [ ]:
# Point to a local or custom OpenAI-compatible API server
llm_custom = OpenAIChatModel(
    model_name="my-local-model",
    base_url="http://localhost:8000/v1",
    api_key="not-needed",  # some local servers don't require a key
    temperature=0.7,
)

# Usage is identical to the standard model
# response = llm_custom.run("Hello, local model!")
# print(response)

## 7. Using LLMs with Agents

DemoGPT's LLM wrappers are designed to plug directly into DemoGPT agents. Here is a quick example of passing an LLM to a `ToolCallingAgent`, which can use tools and reason about tasks.

In [ ]:
from demogpt_agenthub.llms import OpenAIChatModel
from demogpt_agenthub.agents import ToolCallingAgent

# 1. Create the LLM
llm = OpenAIChatModel(model_name="gpt-4o-mini", temperature=0.0)

# 2. Define tools (empty list for this demo; see the Tools notebook for details)
tools = []

# 3. Create an agent with the LLM
agent = ToolCallingAgent(tools=tools, llm=llm, verbose=True)

# 4. Run the agent
result = agent.run("What are three benefits of using Python for data science?")
print(result)

## 8. Summary

Here are the key takeaways from this notebook:

- **`OpenAIChatModel`** is the primary wrapper for OpenAI chat models (GPT-4.1, GPT-4o, GPT-4o-mini, etc.). Use `.run(prompt)` to get a text response.
- **`OpenAIModel`** is available for the legacy text completion model `gpt-3.5-turbo-instruct`, but this endpoint is deprecated. Prefer `OpenAIChatModel` for all new projects.
- Both classes accept standard parameters such as `temperature`, `max_tokens`, and `api_key`.
- You can point any model wrapper at a custom `base_url` for self-hosted or proxy endpoints.
- LLM instances plug directly into DemoGPT agents like `ToolCallingAgent` via the `llm` parameter.
- All wrappers extend `BaseLLM`, ensuring a consistent `.run()` interface across different model backends.